# PROTAC Training Pipeline: Antibacterial Protein Degraders

This notebook trains ML models to predict antibacterial activity of PROTACs (Proteolysis-Targeting Chimeras) and peptidomimetics.

**Workflow:**
1. Data Loading (PROTAC-DB or synthetic) → Preprocessing → Featurization
2. Model Training (RandomForest / GradientBoosting)
3. Evaluation
4. Prediction & Ranking

---

## 1. Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from typing import List, Dict, Tuple, Optional

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, classification_report, 
    average_precision_score
)
from imblearn.over_sampling import SMOTE

import joblib

sys.path.append('..')
from modules.protac_module import (
    PROTACGenerator, PROTACFeaturizer, PROTACModel, PROTACPipeline,
    PROTASE_TARGETS, LINKER_TYPES
)
from core.pipeline import UnifiedRanker

## 2. Data Loading

### 2.1 How to Download PROTAC-DB Data

**Steps to obtain the dataset:**

1. Go to [PROTAC-DB](https://protacdb.biohackers.cn) or [PROTAC-DB 2.0](http://protac.must.edu.cn/)
2. Search for antibacterial PROTACs or download full dataset
3. Export as CSV with: SMILES, activity (IC50/MIC), target protein
4. Save to `data/protac/training_data.csv`

**Target bacterial proteins for antibacterial PROTACs:**
- LpxC (Gram-negative)
- ClpP (Gram-positive)
- FtsH (Gram-negative)
- DegP (Gram-negative)

In [ ]:
DATA_DIR = Path('../data/protac')
RESULTS_DIR = Path('../results/protac')
MODELS_DIR = Path('../models')

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def load_protac_data(filepath: str) -> pd.DataFrame:
    """Load PROTAC training data."""
    filepath = Path(filepath)
    
    if not filepath.exists():
        print(f"⚠️  File not found: {filepath}")
        print("Generating synthetic data for demonstration...")
        return generate_synthetic_protac_data(n_samples=2000)
    
    return pd.read_csv(filepath)

In [ ]:
from modules.protac_module import generate_synthetic_protac_data

df = load_protac_data(DATA_DIR / 'training_data.csv')
print(f"Loaded {len(df)} compounds")
df.head()

## 3. Data Preprocessing

In [ ]:
def preprocess_protac_data(df: pd.DataFrame, 
                          mw_threshold: float = 1000) -> pd.DataFrame:
    """Preprocess PROTAC dataset."""
    df_clean = df.copy()
    
    if 'smiles' not in df_clean.columns:
        raise ValueError("Dataset must contain 'smiles' column")
    
    if 'MW' in df_clean.columns:
        df_clean = df_clean[df_clean['MW'] <= mw_threshold]
    
    df_clean = df_clean.dropna(subset=['smiles', 'activity'])
    df_clean = df_clean.drop_duplicates(subset=['smiles'])
    
    return df_clean.reset_index(drop=True)

In [ ]:
df_processed = preprocess_protac_data(df)
print(f"Processed {len(df_processed)} compounds")
print(f"\nClass distribution:")
print(df_processed['activity'].value_counts())

## 4. Featurization

In [ ]:
print("Featurizing compounds...")
features_df = PROTACFeaturizer.featurize_batch(df_processed['smiles'].tolist())

if 'activity' in df_processed.columns:
    features_df['activity'] = df_processed['activity'].values

print(f"Generated {len(features_df.columns)} features")
features_df.head()

## 5. Model Training

In [ ]:
feature_cols = [col for col in features_df.columns if col not in ['smiles', 'activity', 'name']]
X = features_df[feature_cols].values
y = df_processed['activity'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
print(f"After SMOTE: {len(X_train_balanced)} samples")

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_balanced, y_train_balanced)

y_pred = rf_model.predict(X_test_scaled)
y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

print("Random Forest Results:")
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

## 6. Evaluation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

from sklearn.metrics import roc_curve, precision_recall_curve

roc_curve_data = roc_curve(y_test, y_prob)
axes[0].plot(roc_curve_data[0], roc_curve_data[1], 'b-', label=f'RF (AUC={roc_auc_score(y_test, y_prob):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

pr_curve_data = precision_recall_curve(y_test, y_prob)
axes[1].plot(pr_curve_data[0], pr_curve_data[1], 'r-', label=f'AP={average_precision_score(y_test, y_prob):.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'evaluation_metrics.png', dpi=150)
plt.show()

## 7. Prediction & Ranking

In [ ]:
pipeline = PROTACPipeline(output_dir=RESULTS_DIR)
pipeline.model = rf_model
pipeline.model.scaler = scaler
pipeline.model.feature_cols = feature_cols

In [ ]:
warhead_example = "CC(C)CC1=CC=C(C=C1)C(=O)O"
generated_protacs = pipeline.generate_degraders(warhead_example, n_analogs=50)
print(f"Generated {len(generated_protacs)} PROTAC candidates")

predictions = pipeline.screen_degraders(generated_protacs)
print("\nTop PROTAC Candidates:")
predictions[['smiles', 'probability_active', 'degrader_score', 'viability_score']].head(10)

## 8. Save Model & Results

In [ ]:
import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

model_data = {
    'model': rf_model,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'model_type': 'RandomForest',
    'training_date': timestamp,
    'metrics': {
        'auc_roc': float(roc_auc_score(y_test, y_prob)),
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'f1': float(f1_score(y_test, y_pred))
    }
}

model_path = MODELS_DIR / f'protac_model_{timestamp}.joblib'
joblib.dump(model_data, model_path)
print(f"Model saved to: {model_path}")

predictions_path = RESULTS_DIR / f'protac_predictions_{timestamp}.csv'
predictions.to_csv(predictions_path, index=False)
print(f"Predictions saved to: {predictions_path}")

## 9. Multi-Category Integration Test

In [ ]:
from core.pipeline import get_category_predictions

print("Testing multi-category unified ranking...")

dfs = {
    'amp': get_category_predictions('amp', 50),
    'repurposed': get_category_predictions('repurposed', 50),
    'polyphenol': get_category_predictions('polyphenol', 50),
    'protac': predictions
}

weights = {
    'amp': 1.2,
    'repurposed': 1.0,
    'polyphenol': 1.0,
    'protac': 1.3
}

combined_results = UnifiedRanker.rank_multi_category(dfs, weights, top_k=30)

print(f"\nTotal candidates: {len(combined_results)}")
print("\nCategory distribution:")
print(combined_results['category'].value_counts())

combined_results[['category', 'final_score', 'activity_score', 'druglikeness_score']].head(15)

## Summary

This notebook demonstrates:
1. **PROTAC generation** - Three-part design (warhead-linker-E3 ligase)
2. **Degrader-specific featurization** - Amide bonds, chain length, branching
3. **Model training** - RandomForest for activity prediction
4. **Multi-category ranking** - Unified ranking across AMPs, repurposed drugs, polyphenols, PROTACs

**Next steps:**
- Load real PROTAC-DB data
- Add docking for target validation
- Integrate with core pipeline for composite scoring